## COCO Threat Dataset Conversion to YOLO Format

In [1]:
import os
import json
import shutil
from tqdm import tqdm

base_dir = "../datasets/coco"
output_dir = "../datasets/coco-threat"

target_classes = {8, 27, 33, 39, 49, 87}
coco_id_to_yolo_id = {coco_id: i for i, coco_id in enumerate(sorted(target_classes))}
splits = ['train', 'val']

In [2]:
def coco_to_yolo(bbox, img_w, img_h):
    x, y, w, h = bbox
    x_c = (x + w / 2) / img_w
    y_c = (y + h / 2) / img_h
    return [x_c, y_c, w / img_w, h / img_h]

In [ ]:
# Perform the conversion for each split
def process_split(split):
    print(f"Processing {split} split...")
    
    # Grabs and creates the necessary directories
    anno_path = os.path.join(base_dir, "annotations", f"instances_{split}2017.json")
    image_dir = os.path.join(base_dir, "images", f"{split}2017")
    out_img_dir = os.path.join(output_dir, "images", split)
    out_lbl_dir = os.path.join(output_dir, "labels", split)
    
    with open(anno_path) as f:
        coco = json.load(f)
    
    
    imgs = {img["id"]: img for img in coco["images"]}
    anns = [ann for ann in coco["annotations"] if ann["category_id"] in target_classes]
    
    
    labels_by_image = {}
    for ann in anns:
        img_id = ann["image_id"]
        img = imgs[img_id]
        file_name = img["file_name"]
        img_w, img_h = img["width"], img["height"]
        
        # Convert COCO bbox to YOLO format
        # YOLO format: [x_center, y_center, width, height]
        yolo_box = coco_to_yolo(ann["bbox"], img_w, img_h)
        
        #Map COCO category ID to YOLO class ID
        yolo_class = coco_id_to_yolo_id[ann["category_id"]]
        
        # Create the label line
        # YOLO format: class_id x_center y_center width height where all values are normalized to [0, 1]
        label_line = [yolo_class] + yolo_box
        
        if file_name not in labels_by_image:
            labels_by_image[file_name] = []
        labels_by_image[file_name].append(label_line)
    
    # For each image, create a corresponding label file
    for file_name, labels in tqdm(labels_by_image.items()):
        src_img = os.path.join(image_dir, file_name)
        dst_img = os.path.join(out_img_dir, file_name)
        dst_txt = os.path.join(out_lbl_dir, file_name.replace(".jpg", ".txt"))
        
        if not os.path.exists(src_img):
            continue
        
        # Copies files to the output directory
        shutil.copyfile(src_img, dst_img)
        
        # Writes the labels to a text file
        with open(dst_txt, "w") as f:
            for label in labels:
                f.write(" ".join([f"{x:.6f}" for x in label]) + "\n")

In [ ]:
for split in splits:
    process_split(split)

print("✅ Done: COCO subset converted to YOLO format.")

Processing train split...


 12%|█▏        | 2440/20325 [00:02<00:15, 1191.28it/s]


KeyboardInterrupt: 

In [ ]:
import os

# Mapping COCO-threat IDs to OpenImages IDs for combination
coco_to_openimages_mapping = {
    0: 0,    # truck
    1: 9,    # backpack
    2: 12,   # suitcase
    3: 1,    # baseball bat
    4: 4,    # knife
    5: 13,   # scissors
}

# Input directories
input_train_labels_dir = "/Users/lvntkymn14/Documents/Bachelors_Thesis_Project_Files/Thesis/yolov5/datasets/coco-threat/labels/train/"
input_val_labels_dir = "/Users/lvntkymn14/Documents/Bachelors_Thesis_Project_Files/Thesis/yolov5/datasets/coco-threat/labels/val/"

# Output directories
output_train_labels_dir = "/Users/lvntkymn14/Documents/Bachelors_Thesis_Project_Files/Thesis/yolov5/datasets/coco-threat/labels_remapped/train/"
output_val_labels_dir = "/Users/lvntkymn14/Documents/Bachelors_Thesis_Project_Files/Thesis/yolov5/datasets/coco-threat/labels_remapped/val/"

# Make sure output folders exist
os.makedirs(output_train_labels_dir, exist_ok=True)
os.makedirs(output_val_labels_dir, exist_ok=True)

# Function to remap labels from COCO IDs to OpenImages IDs
def remap_labels(input_dir, output_dir):
    for filename in os.listdir(input_dir):
        if not filename.endswith(".txt"):
            continue
        
        input_path = os.path.join(input_dir, filename)
        output_path = os.path.join(output_dir, filename)
        
        with open(input_path, "r") as f:
            lines = f.readlines()
        
        new_lines = []
        for line in lines:
            parts = line.strip().split()
            if len(parts) != 5:
                continue
            try:
                old_class_id = int(float(parts[0]))
            except ValueError:
                print(f"Skipping badly formatted line in {filename}: {line.strip()}")
                continue

            if old_class_id not in coco_to_openimages_mapping:
                continue
            
            new_class_id = coco_to_openimages_mapping[old_class_id]
            rest_of_line = " ".join(parts[1:])
            new_line = f"{new_class_id} {rest_of_line}"
            new_lines.append(new_line)
        
        # Only write to the output file if there are new lines
        if new_lines:
            with open(output_path, "w") as f:
                f.write("\n".join(new_lines))

# Remap labels for both train and val splits
remap_labels(input_train_labels_dir, output_train_labels_dir)
remap_labels(input_val_labels_dir, output_val_labels_dir)
print("Finished remapping COCO-threat labels into train/val splits!")

🚀 Remapping COCO-threat train labels...
🚀 Remapping COCO-threat val labels...
✅ Finished remapping COCO-threat labels into train/val splits!
